# Step 3: Operationalize with an ML pipeline and model registry

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

In this step you automate the workflow of the previous notebooks with [SageMaker Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html). The pipeline processes the data, trains and evaluates the model, and registers it in the model registry if its AUC meets a threshold.

You build the pipeline with the [`@step` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator.html), which turns the `pipeline_steps` functions into pipeline steps without code changes. `@step` is to pipelines what `@remote` was to jobs in notebook 02. At the end of the notebook you see how the `ModelTrainer` from notebook 02 becomes a native `TrainingStep`, the pipeline counterpart of script mode.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI jobs and SageMaker SDK ||
|3. |Operationalize with ML pipeline, model registry |**<<<< YOU ARE HERE**|
|4. |Deploy the model and add data and model monitoring ||
|5. |Automate model deployment with a SageMaker project ||

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

In [ ]:
%store -r 

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

---

## Setup

In [ ]:
import json
import os
import boto3
import mlflow
from time import gmtime, strftime, sleep

# Pipeline orchestration is in sagemaker.mlops, the workflow primitives are in sagemaker.core
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import CacheConfig
from sagemaker.mlops.workflow.fail_step import FailStep
from sagemaker.mlops.workflow.condition_step import ConditionStep
from sagemaker.mlops.workflow.function_step import step
from sagemaker.core.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.core.workflow.functions import Join
from sagemaker.core.workflow.execution_variables import ExecutionVariables
from sagemaker.core.workflow.pipeline_definition_config import PipelineDefinitionConfig

from IPython.display import HTML, display

### Set constants

In [ ]:
# Set names of pipeline objects, experiment, and a model
project = "from-idea-to-prod"

current_timestamp = strftime('%d-%H-%M-%S', gmtime())

# Reuse the single MLflow experiment and registered model shared across all
# notebooks (created in notebook 01 and restored via %store). Fall back to
# the same date-suffixed (-DDMMYY) names when running this notebook standalone.
date_suffix = strftime('%d%m%y', gmtime())
if "experiment_name" not in dir():
    experiment_name = f"from-idea-to-prod-experiment-{date_suffix}"
    %store experiment_name
if "registered_model_name" not in dir():
    registered_model_name = f"from-idea-to-prod-model-{date_suffix}"
    %store registered_model_name

# A stable name: each run of this notebook updates the same pipeline
pipeline_name = f"{project}-pipeline"

model_package_group_name = registered_model_name

model_approval_status = "PendingManualApproval"

In [ ]:
# Set instance types and counts
process_instance_type = "ml.r5.xlarge"
train_instance_type = "ml.r5.xlarge"

In [ ]:
# Input data S3 url (from 00-start-here.ipynb)
input_s3_url = s3_data_uri

# Set S3 urls for various datasets produced in the pipeline
output_s3_prefix = f"s3://{bucket_name}/{bucket_prefix}"
output_s3_url = f"{output_s3_prefix}/output"

train_s3_url = f"{output_s3_prefix}/train"
validation_s3_url = f"{output_s3_prefix}/validation"
test_s3_url = f"{output_s3_prefix}/test"
baseline_s3_url = f"{output_s3_prefix}/baseline"

# Used by the optional notebook additional-topics/04-monitoring-sagemaker-native
prediction_baseline_s3_url = f"{output_s3_prefix}/prediction_baseline"


In [ ]:
# Use the custom XGBoost image built in 00-start-here.ipynb
XGBOOST_IMAGE_URI = xgboost_image
print(f"XGBoost image: {XGBOOST_IMAGE_URI}")

In [ ]:
%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url
%store pipeline_name
%store model_package_group_name
%store prediction_baseline_s3_url
%store output_s3_url

In [ ]:
print(f"Train S3 url: {train_s3_url}")
print(f"Validation S3 url: {validation_s3_url}")
print(f"Test S3 url: {test_s3_url}")
print(f"Data baseline S3 url: {baseline_s3_url}")


### Configure MLflow

In [ ]:
# Wait for MLflow app to be ready
sm_client = boto3.client("sagemaker")

while sm_client.describe_mlflow_app(Arn=mlflow_arn)['Status'] not in ['Created', 'Updated']:
    print(f"Waiting for MLflow app...")
    sleep(10)

print(f"MLflow app is ready: {mlflow_arn}")

In [ ]:
# MLflow prints "View run ... at: https://mlflow.sagemaker.<region>.app.aws/..." when a run
# ends. That host is the SageMaker MLflow *tracking API* endpoint, not the UI, so the links
# do not open. Suppress them; the notebook shows working presigned UI links instead.
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
mlflow.set_tracking_uri(mlflow_arn)
experiment = mlflow.set_experiment(experiment_name=experiment_name)

### Configure the defaults of the `@step` decorator
A [configuration file](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator-config.html) sets the defaults of all `@step` and `@remote` calls: the container image, the instance type, the environment variables, and which local files to upload. You don't need to repeat these settings for each step.

The next cells write two files to the workshop folder:
- `requirements.txt`: the packages that each step installs at start
- `config.yaml`: the defaults

Then they copy `config.yaml` to the user-level configuration folder of the SageMaker Python SDK, `~/.config/sagemaker/`.

<div class="alert alert-warning">The SDK applies the user-level configuration to every <code>@step</code> and <code>@remote</code> call in this JupyterLab space, also in other notebooks. Arguments that you pass in the code, as in notebook 02, override the defaults. To go back to the SDK defaults, delete <code>~/.config/sagemaker/config.yaml</code>.</div>

In [ ]:
local_training_dir = "./"
os.makedirs(local_training_dir, exist_ok=True)
with open(os.path.join(local_training_dir, "requirements.txt"), "w") as file_handle:
    file_handle.write("sagemaker>=3,<4")

In [ ]:
# config.yaml sets defaults for @step decorator.
# MLflow reads MLFLOW_TRACKING_URI and MLFLOW_EXPERIMENT_NAME natively from env.
config_yaml = f"""SchemaVersion: '1.0'
SageMaker:
  PythonSDK:
    Modules:
      RemoteFunction:
        InstanceType: ml.r5.xlarge
        Dependencies: ./requirements.txt
        ImageUri: '{XGBOOST_IMAGE_URI}'
        IncludeLocalWorkDir: true
        EnvironmentVariables: {{'MLFLOW_TRACKING_URI': '{mlflow_arn}', 'MLFLOW_EXPERIMENT_NAME': '{experiment_name}', 'CONTAINER_IMAGE': '{XGBOOST_IMAGE_URI}'}}
        CustomFileFilter:
          IgnoreNamePatterns:
            - "data"
            - "tmp"
            - "img"
            - "design"
            - "assignments"
            - "additional-topics"
            - ".git"
            - "*.ipynb"
            - "*.md"
            - "__pycache__"
"""

with open('config.yaml', 'w') as f:
    f.write(config_yaml)

# Also set locally so local function calls work the same way
os.environ['MLFLOW_TRACKING_URI'] = mlflow_arn
os.environ['MLFLOW_EXPERIMENT_NAME'] = experiment_name
os.environ['CONTAINER_IMAGE'] = 'local'

print(config_yaml)

In [ ]:
from platformdirs import user_config_dir

config_dir = user_config_dir("sagemaker")
os.makedirs(config_dir, exist_ok=True)
!cp config.yaml {os.path.join(config_dir, 'config.yaml')}
print(f"Config copied to {config_dir}")

---

## Part 1: Test the steps in the notebook
Before you build the pipeline, call each step function in the notebook. This is the same code that runs in the pipeline.

In [ ]:
from pipeline_steps.preprocess import preprocess
from pipeline_steps.train_fn import train
from pipeline_steps.evaluate import evaluate
from pipeline_steps.register import register

In [ ]:
# Create a parent MLflow run to group all local test steps
mlflow.set_tracking_uri(mlflow_arn)
mlflow.set_experiment(experiment_name)
local_parent_run = mlflow.start_run(run_name=f"local-{current_timestamp}")
local_run_id = local_parent_run.info.run_id
mlflow.end_run()  # End so child steps can resume it
print(f"Parent run: local-{current_timestamp} ({local_run_id})")

### Preprocess

In [ ]:
r_preprocess = preprocess(
    input_data_s3_path=input_s3_url,
    output_s3_prefix=output_s3_prefix,
    pipeline_run_id=local_run_id,
)
r_preprocess

### Train
Call `train()` in the notebook. It loads the data from S3, trains the XGBoost model, saves it to S3, and logs it in MLflow.

In [ ]:
r_train = train(
    train_data_s3_path=r_preprocess['train_data'],
    validation_data_s3_path=r_preprocess['validation_data'],
    output_s3_prefix=output_s3_prefix,
    pipeline_run_id=r_preprocess['pipeline_run_id'],
)
r_train

### Evaluate

In [ ]:
r_eval = evaluate(
    test_x_data_s3_path=r_preprocess['test_x_data'],
    test_y_data_s3_path=r_preprocess['test_y_data'],
    model_s3_path=r_train['model_s3_path'],
    output_s3_prefix=output_s3_prefix,
    pipeline_run_id=r_preprocess['pipeline_run_id'],
)
r_eval

### Register
The register step uses the same pattern as notebook 02. The MLflow artifact store stays the single source of truth for the model files:

1. It logs [`pipeline_steps/inference.py`](pipeline_steps/inference.py) into the artifact store of the logged model, under `code/`
2. It logs the inference specification on the MLflow model: the container image, the model data in the MLflow artifact store, and the environment variables `SAGEMAKER_PROGRAM` and `SAGEMAKER_SUBMIT_DIRECTORY`
3. It calls `mlflow.register_model()`. Because the specification exists already, the model package in the SageMaker Model Registry is deployable.

In [ ]:
print(f"Registering model to: {model_package_group_name}")

r_register = register(
    model_package_group_name=model_package_group_name,
    model_approval_status=model_approval_status,
    evaluation_result=r_eval['evaluation_result'],
    mlflow_run_id=r_train['mlflow_run_id'],
    training_job_name=r_train['training_job_name'],
    pipeline_run_id=r_preprocess['pipeline_run_id'],
)
r_register


### Check the model package in the SageMaker Model Registry
Because the MLflow app uses `AutoModelRegistrationEnabled`, `mlflow.register_model()` also created a model package version in the SageMaker Model Registry. The next cell reads it.

In [ ]:
model_package = sm_client.describe_model_package(ModelPackageName=r_register["model_package_arn"])
print(f"Model package:   {model_package['ModelPackageArn']}")
print(f"Approval status: {model_package['ModelApprovalStatus']}")
print(f"Deployable:      {'InferenceSpecification' in model_package}")

# Notebooks 04 and 05 deploy model versions from this model package group
target_model_package_group_name = r_register['model_package_group_name']

In [ ]:
%store target_model_package_group_name 

---

## Part 2: Build a pipeline with the `@step` decorator
All steps work in the notebook, so wrap them with `@step` to build the pipeline.

### Pipeline parameters
[Pipeline parameters](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-parameters.html) let you change the input data, the instance types, the approval status, and the AUC threshold for each pipeline execution without changing the pipeline definition.

In [ ]:
input_s3_url_param = ParameterString(
    name="InputDataUrl",
    default_value=input_s3_url
)

process_instance_type_param = ParameterString(
    name="ProcessingInstanceType",
    default_value=process_instance_type
)
train_instance_type_param = ParameterString(
    name="TrainingInstanceType",
    default_value=train_instance_type
)

model_approval_status_param = ParameterString(
    name="ModelApprovalStatus",
    default_value=model_approval_status
)

test_score_threshold_param = ParameterFloat(
    name="TestScoreThreshold",
    default_value=0.7
)

model_package_group_name_param = ParameterString(
    name="ModelPackageGroupName",
    default_value=model_package_group_name
)


### Build the pipeline
The next cell wraps the four functions that you tested with `@step`. A condition step registers the model only if its AUC meets the threshold, otherwise a fail step stops the pipeline.

SageMaker Pipelines builds the directed acyclic graph (DAG) of the pipeline from the **data dependencies between the steps**. When you pass the output of one step to another step, the second step runs after the first. Steps without data dependencies run in parallel.


In [ ]:
cache_config = CacheConfig(enable_caching=True, expire_after="p30d")

# Preprocess step
step_preprocess = step(
    preprocess,
    name=f"{project}-preprocess",
    instance_type=process_instance_type_param,
    keep_alive_period_in_seconds=3600,
)(
        input_data_s3_path=input_s3_url_param,
        output_s3_prefix=output_s3_prefix,
        pipeline_run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
)

# Train step
step_train = step(
    train,
    name=f"{project}-train",
    instance_type=train_instance_type_param,
    keep_alive_period_in_seconds=3600,
)(
        train_data_s3_path=step_preprocess['train_data'],
        validation_data_s3_path=step_preprocess['validation_data'],
        output_s3_prefix=output_s3_prefix,
        pipeline_run_id=step_preprocess['pipeline_run_id'],
)

# Evaluate step
step_evaluate = step(
    evaluate,
    name=f"{project}-evaluate",
    instance_type=process_instance_type_param,
    keep_alive_period_in_seconds=3600,
)(
        test_x_data_s3_path=step_preprocess['test_x_data'],
        test_y_data_s3_path=step_preprocess['test_y_data'],
        model_s3_path=step_train['model_s3_path'],
        output_s3_prefix=output_s3_prefix,
        pipeline_run_id=step_preprocess['pipeline_run_id'],
)

# Register step — uses mlflow.register_model() with auto-sync to SageMaker
step_register = step(
    register,
    name=f"{project}-register",
    keep_alive_period_in_seconds=3600,
)(
        model_package_group_name=model_package_group_name_param,
        model_approval_status=model_approval_status_param,
        evaluation_result=step_evaluate['evaluation_result'],
        mlflow_run_id=step_train['mlflow_run_id'],
        training_job_name=step_train['training_job_name'],
        pipeline_run_id=step_preprocess['pipeline_run_id'],
)

# Fail step
step_fail = FailStep(
    name=f"{project}-fail",
    error_message=Join(on=" ", values=["Execution failed due to AUC Score <", test_score_threshold_param]),
)

# Condition step
step_conditional_register = ConditionStep(
    name=f"{project}-check-metrics",
    conditions=[ConditionGreaterThanOrEqualTo(
        left=step_evaluate['evaluation_result']['classification_metrics']['auc_score']['value'],
        right=test_score_threshold_param,
    )],
    if_steps=[step_register],
    else_steps=[step_fail],
)

# Create the pipeline
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        input_s3_url_param,
        process_instance_type_param,
        train_instance_type_param,
        model_approval_status_param,
        test_score_threshold_param,
        model_package_group_name_param,
    ],
    steps=[step_conditional_register],
    pipeline_definition_config=PipelineDefinitionConfig(use_custom_job_prefix=True),
)


The pipeline is defined. Its four steps call the same functions that you tested in the notebook.

### Create and run the pipeline
`pipeline.upsert()` creates the pipeline or updates its definition. `pipeline.start()` starts an execution.

In [ ]:
pipeline.upsert(role_arn=sm_role)

In [ ]:
pipeline_execution = pipeline.start()
pipeline_execution.describe()

In [ ]:
display(
    HTML('<b>See <a target="_blank" rel="noopener noreferrer" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/executions/{}/graph">the pipeline execution</a> in the Studio UI</b>'.format(
        domain_id, region, pipeline_name, pipeline_execution.describe()['PipelineExecutionArn'].split('/')[-1]
    ))
)

In [ ]:
%%time
# Wait for the pipeline execution to complete, about 15 minutes
pipeline_execution.wait()
pipeline_execution.list_steps()

The Studio UI shows the pipeline graph and the status of each step. Select a step to see its details, logs, and outputs:

![Pipeline execution graph in Studio with the preprocess, train, evaluate, check-metrics, register, and fail steps](img/pipeline-execution-graph.png)

Each pipeline execution is also a parent run in MLflow, with a child run for each step. Run the next cells and choose the links to see the runs in the MLflow UI.

In [ ]:
from html import escape

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']
experiment_runs_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/runs?workspace=default"

# Open the MLflow UI first: the presigned URL authenticates your browser session
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    '1. Open the MLflow UI and authenticate</a></b>'
))

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(experiment_runs_link, quote=True)}">'
    '2. Open the runs of the experiment</a></b>'
))

---

## Beyond `@step`: native pipeline step types
SageMaker Pipelines also has [native step types](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps-types.html), such as `ProcessingStep`, `TrainingStep`, `TransformStep`, and `TuningStep`. A native step runs a SageMaker AI job with an explicit configuration of the container, the infrastructure, and the input and output channels. The two notebook 02 options map to the two kinds of steps:

| | `@step` decorator | Native step types |
|---|---|---|
| **Notebook 02 counterpart** | `@remote` | `ModelTrainer` in script mode |
| **Code** | The functions that you tested in the notebook | Entry scripts, such as `train_fn.py` with its `main()` |
| **Container control** | One image for the step function | Any image per step: your own, framework, or built-in |
| **Data wiring** | Function return values | Step properties, such as `step.properties.ModelArtifacts.S3ModelArtifacts` |
| **SageMaker AI job features** | Basics | Spot training, checkpoints, warm pools, distributed training, hyperparameter tuning |
| **Best for** | Fast iteration, light logic such as evaluation gates and registration | Production training and processing |

Many teams combine both: native steps for compute-heavy training and processing, `@step` for the light logic. A `@step` function can consume the output of a native step through its properties.

### From `ModelTrainer` to `TrainingStep`
To use the `ModelTrainer` from notebook 02 in a pipeline, create it with a `PipelineSession`. With this session, `trainer.train()` doesn't start a job. It returns the arguments of the job, and a `TrainingStep` runs them when the pipeline runs. The next cell builds such a step and shows its definition. It doesn't start a job.

In [ ]:
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.steps import TrainingStep
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, SourceCode, Compute

# Same ModelTrainer as in notebook 02, with a PipelineSession
trainer = ModelTrainer(
    training_image=XGBOOST_IMAGE_URI,
    source_code=SourceCode(
        source_dir="pipeline_steps",
        entry_script="train_fn.py",
        ignore_patterns=["__pycache__", ".ipynb_checkpoints"],
    ),
    compute=Compute(instance_type=train_instance_type, instance_count=1),
    hyperparameters={"output_s3_prefix": output_s3_prefix},
    base_job_name=f"{project}-training",
    role=sm_role,
    environment={
        "MLFLOW_TRACKING_URI": mlflow_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "CONTAINER_IMAGE": XGBOOST_IMAGE_URI,
    },
    sagemaker_session=PipelineSession(),
)

# With a PipelineSession, train() returns the job arguments instead of starting a job
step_train_native = TrainingStep(
    name=f"{project}-train-native",
    step_args=trainer.train(
        input_data_config=[
            InputData(channel_name="train", data_source=train_s3_url),
            InputData(channel_name="validation", data_source=validation_s3_url),
        ]
    ),
)

native_pipeline = Pipeline(name=f"{project}-native-pipeline", steps=[step_train_native])
step_definition = json.loads(native_pipeline.definition())["Steps"][0]
print(f"Step type:      {step_definition['Type']}")
print(f"Job arguments:  {', '.join(step_definition['Arguments'])}")
print(f"Model artifact: {step_train_native.properties.ModelArtifacts.S3ModelArtifacts.expr}")

In a pipeline, you pass `step_train_native.properties.ModelArtifacts.S3ModelArtifacts` to the next step, for example to the `evaluate()` function in a `@step`. This notebook doesn't run the native pipeline.

To go deeper:
- [Pipeline steps](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps.html): step properties, data dependencies, and parallel steps
- [Add a step](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps-types.html): requirements and examples for each native step type
- [Lift-and-shift Python code with the @step decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator.html): best practices and limitations
- [@step decorator sample notebooks](https://github.com/aws/amazon-sagemaker-examples/tree/main/sagemaker-pipelines/step-decorator)

---

## Optional: extend the pipeline
The following optional notebooks extend this pipeline:
- [Quality checks and batch transform](additional-topics/03b-quality-checks-batch-transform.ipynb): add data quality checks, model quality checks, and batch transform to the pipeline
- [Feature Store integration](additional-topics/03c-feature-store.ipynb): manage the features in SageMaker Feature Store and use them in the pipeline

---

## Summary
In this notebook you:
- tested the `preprocess()`, `train()`, `evaluate()`, and `register()` functions in the notebook
- built a SageMaker pipeline from the same functions with the `@step` decorator, with a condition that registers the model only if its AUC meets the threshold
- registered a deployable model version in the MLflow Model Registry and the SageMaker Model Registry
- saw how the `ModelTrainer` from notebook 02 runs as a native `TrainingStep`

The model package group now has a model version with the status `PendingManualApproval`. In step 4 you approve it, deploy it to an endpoint, and monitor it.

## Continue with step 4
Open the step 4 notebook [04-deployment-and-monitoring](04-deployment-and-monitoring.ipynb). In step 4 you deploy the approved model and monitor it. In step 5 you automate the deployment with a CI/CD pipeline.

## Further development ideas for your real-world projects
- Add [bias detection and model explainability](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps.html#step-type-clarify-check) steps. Add model metrics calculated by [SageMaker Clarify](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-configure-processing-jobs.html) to the model metadata in the model registry
- Add event-driven launching of the ML pipeline as soon as a new dataset is uploaded to an Amazon S3 bucket. You can use [Amazon EventBridge integration](https://docs.aws.amazon.com/sagemaker/latest/dg/pipeline-eventbridge.html#pipeline-eventbridge-schedule) to implement various event-driven workflows
- Use a designated IAM execution role for the pipeline execution
- Add data encryption by using S3 bucket encryption and AWS KMS keys for container EBS volume encryption

## Additional resources
- [Automate Machine Learning Workflows](https://aws.amazon.com/getting-started/hands-on/machine-learning-tutorial-mlops-automate-ml-workflows/)
- [Amazon SageMaker Feature Store workshop](https://github.com/aws-samples/amazon-sagemaker-feature-store-end-to-end-workshop)
- [SageMaker Pipelines steps documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps.html)
- [@step decorator sample notebooks](https://github.com/aws/amazon-sagemaker-examples/tree/main/sagemaker-pipelines/step-decorator)

## Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>